In [61]:
import polars as pl
import polars_st as st
import requests
import subprocess

# from palettable.cartocolors.qualitative import Prism_8
from palettable.colorbrewer.qualitative import Set3_11
import shutil

TIPPECANOE = shutil.which("tippecanoe")
assert TIPPECANOE, "tippecanoe not found on PATH"

cfg = pl.Config()
cfg.set_tbl_rows(25)

palette: list[str] = Set3_11.hex_colors

In [62]:
# https://data.cityofnewyork.us/Transportation/Parking-Regulation-Locations-and-Signs/nfid-uabd/about_data
df_prs = pl.read_csv("../datasets/prs.csv")

In [63]:
# they are all current i think
df_prs = (
    df_prs.with_columns(
        pl.when(pl.col("support") == "SAME")
        .then(None)
        .otherwise(pl.col("support"))
        .forward_fill()
        .name.keep()
    )
    # "POLE",
    .filter(
        (pl.col("record_type") == "Current")
        & (
            pl.col("support").is_in(["DR"])
            & (pl.col("sign_x_coord").is_not_null())
            & (pl.col("sign_y_coord").is_not_null())
        )
    )
    .with_columns(
        pl.col("support"),
        # .replace_strict({"DR": "Drive Rail"}).name.keep(),
        geometry=st.point(pl.concat_arr("sign_x_coord", "sign_y_coord"))
        .st.set_srid(2263)
        .st.to_srid(4326),
    )
).select(
    "on_street",
    "on_street_suffix",
    "from_street",
    "to_street",
    "side_of_street",
    "distance_from_intersection",
    # so we can cluster with pmtile easier
    pl.col("support").alias("category"),
    st.geom(),
)

In [64]:
# From https://data.cityofnewyork.us/Transportation/Bicycle-Parking/592z-n7dk/about_data
df_bp = st.read_file("../datasets/bike_parking.geojson")

df_bp = df_bp.select(
    st.geom(),
    pl.col("racktype").alias("category"),
)

In [65]:
# From https://data.cityofnewyork.us/Transportation/Bike-Parking-Shelter/dimy-qyej/about_data
res = requests.get("https://data.cityofnewyork.us/resource/dimy-qyej.geojson").content
df_shelters = st.read_file(res).select(st.geom(), pl.lit("Shelter").alias("category"))

In [66]:
df_bp = pl.concat([df_bp, df_shelters, df_prs], how="diagonal")

In [67]:
category_names = {
    # "WAVE RACK": "Wave Rack",
    # "WAVE RACK (NON DOT)": "Wave Rack",
    # "WAVE RACK (PARKS)": "Wave Rack",
    # "U RACK": "U Rack",
    # "U RACK (DOT)": "U Rack",
    # "SMALL HOOP": "Small Hoop",
    # "LARGE HOOP": "Large Hoop",
    # "DOT SLED (BLACK)": "Sled",
    # "GFI SLED (SILVER)": "Sled",
    # "GFI SLED (SILVER - RIGHT)": "Sled",
    # "BIKE CORRAL": "Bike Corral",
    # "OPAL RACK (PARKS)": "Opal Rack",
    # "STAPLE (PARKS)": "Staple",
    "U Rack": "U-Rack",
    "Undetermined": "Other",
    # bruh
    "Unkown": "Other",
    "Other Non-Dot": "Other",
    "Other Dot": "Other",
    "Byrne Rack": "Other",
    "Dot Sled": "Sled",
    "Gfi Sled": "Sled",
    # Street signs
    "Dr": "Drive Rail",
}
df_bp = df_bp.with_columns(
    pl.col("category")
    .str.replace_all(r"\s*\([^)]*\)\s*", " ")  # remove "(...)" and surrounding spaces
    .str.replace_all(r"\s{2,}", " ")  # collapse any repeated spaces
    .str.strip_chars()
    .str.to_titlecase()
    .replace(category_names)
    .name.keep()
    # pl.col("racktype").str.to_uppercase().replace(rack_mapping).name.keep()
)
# TODO: fix opal rack not being replaced
# df_bp["category"].unique()

In [68]:
unique_categories = df_bp["category"].unique().sort().to_list()

# assign colors deterministically (wrap if more categories than palette)
colors = [palette[i % len(palette)] for i in range(len(unique_categories))]

In [69]:
assert len(unique_categories) <= len(palette), "Not enough colors in palette"

In [70]:
df_category_map = pl.DataFrame(
    {
        "category": unique_categories,
        # "category_id": list(range(len(unique_categories))),
        "color": colors,
    }
)
df_bp = df_bp.join(df_category_map, on="category", how="left")

In [71]:
geojson_files = []
for df_cat in df_bp.partition_by("category"):
    # remove all null cols
    df_cat = df_cat[[s.name for s in df_cat if not (s.null_count() == df_cat.height)]]
    print(df_cat["category"][0], df_cat.shape)

    cat_name = df_cat["category"][0]
    # file_name = f"../out/{cat_name.replace(' ', '_').lower()}.geojson"
    file_name = f"../out/{cat_name}.geojson"

    df_cat.st.write_geojson(file_name)
    geojson_files.append(file_name)

Large Hoop (13887, 3)
U-Rack (7212, 3)
Small Hoop (13502, 3)
Wave Rack (1272, 3)
Bike Corral (831, 3)
Other (182, 3)
Opal Rack (18, 3)
Sled (51, 3)
Staple (1, 3)
Shelter (17, 3)
Drive Rail (248002, 9)


In [74]:
df_bp.select(
    # "racktype",
    "category",
    "color",
    # pl.col("racktype")
    # .str.to_lowercase()
    # .str.replace_all(
    #     r"\W+",
    #     "_",
    # )
    # .alias("id"),
).unique().sort("category").write_json("../src/lib/assets/rack_mapping.json")

In [73]:
# df_bp.st.write_geojson("../out/bike_parking.geojson")
# df_prs.st.write_geojson("../out/parking_signs.geojson")

In [60]:
cmd = [
    TIPPECANOE,
    "-o",
    "../static/data.pmtiles",
    # "-Z 6",
    # "-z 14",
    # "--drop-densest-as-needed",
    "-zg",
    # "-r1",
    # "--cluster-distance=10",
    # "--accumulate-attribute=category:sum",
    "--force",
    "--generate-ids",
    *geojson_files,
    # "../out/bike_parking.geojson",
    # "../out/parking_signs.geojson",
]
res = subprocess.run(cmd, check=True, capture_output=True, text=True)
print("stdout:", res.stdout)
print("stderr:", res.stderr)

stdout: 
stderr: For layer 0, using name "LargeHoop"
For layer 1, using name "URack"
For layer 2, using name "SmallHoop"
For layer 3, using name "WaveRack"
For layer 4, using name "BikeCorral"
For layer 5, using name "Other"
For layer 6, using name "OpalRack"
For layer 7, using name "Sled"
For layer 8, using name "Staple"
For layer 9, using name "Shelter"
For layer 10, using name "DriveRail"
Read 0.00 million features
Read 0.01 million features
Read 0.02 million features
Read 0.03 million features
Read 0.04 million features
Read 0.05 million features
Read 0.06 million features
Read 0.07 million features
Read 0.08 million features
Read 0.09 million features
Read 0.10 million features
Read 0.11 million features
Read 0.12 million features
Read 0.13 million features
Read 0.14 million features
Read 0.15 million features
Read 0.16 million features
Read 0.17 million features
Read 0.18 million features
Read 0.19 million features
Read 0.20 million features
Read 0.21 million features
Read 0.22 m